In [37]:
import torch
from torchvision.datasets import FashionMNIST
from torchvision import transforms, models
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
from torch.utils.data import Subset, ConcatDataset

In [38]:
from torch.utils.data import Dataset
class AdvDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __getitem__(self, index):
        return self.x[index], self.y[index].item()

    def __len__(self):
        return len(self.x)

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [40]:
train_dataset = FashionMNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
train_dataset = Subset(train_dataset, range(30000))
test_dataset = FashionMNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())
# test_dataset = Subset(test_dataset, range(5000)).dataset

print(f'FashionMNIST Train: {len(train_dataset)}, Test: {len(test_dataset)}')

train_dataset_adv = torch.load('./adv_data/resnet_pgd_fashion_train')
train_dataset_adv = Subset(train_dataset_adv, range(30000))
# test_dataset_adv = torch.load('./adv_data/resnet_pgd_cifar_test')
# test_dataset_adv = Subset(test_dataset_adv, range(5000)).dataset
# 
train_dataset = ConcatDataset([train_dataset, train_dataset_adv])
# test_dataset = ConcatDataset([test_dataset, test_dataset_adv])


FashionMNIST Train: 30000, Test: 10000


In [41]:
batch_size = 512

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [42]:
model = models.resnet50(weights="DEFAULT")

In [43]:
model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [44]:
# modify the input and output layers
model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
model.fc = nn.Linear(2048, 10)
model

ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [45]:
lr = 0.001

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
# optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=1e-4)
# optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
best_val_loss = float('inf')
epochs = 10

for epoch in range(epochs):
    # Training
    model.train()
    train_loss = 0
    train_correct = 0
    tarin_bar = tqdm(train_loader, position=0, leave=True)
    for x, y in tarin_bar:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        train_correct += class_pred.eq(y).sum().item()
    train_accuracy = train_correct / len(train_dataset)

    # Validation
    model.eval()
    val_loss = 0
    val_correct = 0
    val_bar = tqdm(test_loader, position=0, leave=True)
    with torch.no_grad():
        for x, y in val_bar:
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = criterion(y_pred, y)
            val_loss += loss.item()
            class_pred = y_pred.argmax(dim=1)
            val_correct += class_pred.eq(y).sum().item()
    val_accuracy = val_correct / len(test_dataset)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), './model/ResNet50_FashionMNIST_pgd.pth')
    print(f'Epoch {epoch+1}/{epochs}, Train Loss: {train_loss/len(train_loader):.6f}, Train Acc: {train_accuracy:.6f}, Val Loss: {val_loss/len(test_loader):.6f}, Val Acc: {val_accuracy:.6f}')
  

  0%|          | 0/118 [00:00<?, ?it/s]


RuntimeError: stack expects each tensor to be equal size, but got [1, 28, 28] at entry 0 and [28, 28] at entry 2

In [ ]:
# model = models.resnet50()
# # modify the input and output layers
# # model.conv1 = nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
# model.fc = nn.Linear(2048, 10)
model.load_state_dict(torch.load('model/ResNet50_FashionMNIST_pgd.pth'))
model = model.to(device)

In [ ]:
model.eval()
val_loss = 0
val_correct = 0
val_bar = tqdm(test_loader, position=0, leave=True)
with torch.no_grad():
    for x, y in val_bar:
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        loss = criterion(y_pred, y)
        val_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        val_correct += class_pred.eq(y).sum().item()
val_accuracy = val_correct / len(test_dataset)

print(f'Val Loss: {val_loss/len(test_loader):.6f}, Val Acc: {val_accuracy:.6f}')

In [ ]:

test_dataset_adv = torch.load('./adv_data/resnet_pgd_fashion_test')

test_loader_adv = DataLoader(test_dataset_adv, batch_size=batch_size, shuffle=False)

In [ ]:
model.eval()
val_loss = 0
val_correct = 0
val_bar = tqdm(test_loader_adv, position=0, leave=True)
with torch.no_grad():
    for x, y in val_bar:
        x, y = x.to(device), y.to(device)
        y_pred = model(x)
        loss = criterion(y_pred, y)
        val_loss += loss.item()
        class_pred = y_pred.argmax(dim=1)
        val_correct += class_pred.eq(y).sum().item()
val_accuracy = val_correct / len(test_dataset)

print(f'Val Loss: {val_loss/len(test_loader):.6f}, Val Acc: {val_accuracy:.6f}')